<a href="https://colab.research.google.com/github/Ali-Hamza-developer/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract
**Lane: Refresh / Content Opportunity Scoring**

Run top to bottom (Runtime → Run all). Requires an `HF_TOKEN` Colab Secret (plain Read type, gated-repositories permission ticked) — see SETUP.md.

In [2]:
# Setup
!pip install -q duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month — never the _sample (June 2026, sealed test month)
print('DuckDB ready, target month =', MONTH)

DuckDB ready, target month = 2026-03


## 1. Unit of analysis + time window

1. **One row =** one pseudonymized content item, for one pseudonymized client, on one report date. Grain key: `report_date + client_hash_id + content_hash_id`.
2. **Table(s):** `fact_content_daily_performance` (features + outcome window), joined to `dim_content` (content metadata) and `dim_clients` (access flags, history start dates).
3. **Time window:** feature window = prior 90 days ending in `month=2026-03`; label/outcome window = the following 30 days. March 2026 is a mid-panel month, not the sealed `_sample` (June 2026) — using the sample here would let the natural outcome window leak into feature development.
4. **Predicting / ranking:** a refresh-priority proxy — `future_decline`, defined as a drop in impressions and/or clicks over the 30-day label window that clears a minimum magnitude and a minimum volume floor (both set and justified in the query cells below, not guessed).
5. **Deliberately excluded:** any rebuilt FlyRank product decision flag (`health_score`, `priority_score`, `action_type`, refresh tiers). These are not shipped in the release, and per the lane guide, treating a rebuilt version of one as a model feature or label causes a circular result — the model just learns to copy the existing rule instead of finding independent signal.

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `impressions_90d` (log) | Feature | observed, prior window only, known at decision time |
| `avg_position_90d` | Feature | observed, prior window |
| `ctr_90d` | Feature | derived from prior-window impressions/clicks |
| `days_since_last_update` | Feature | static content metadata, known at decision time |
| `word_count` | Feature | static content metadata |
| `future_decline` (defined above) | Label/proxy | outcome measured in the 30-day window *after* the decision point |
| `client_hash_id`, `content_hash_id` | Context / join key | grouping and joins only, not a feature |
| `has_gsc_access`, `has_ga4_access` | Context | used to filter for availability, not as a model feature |
| `health_score`, `priority_score`, `action_type` | Excluded | not shipped in the release; rebuilding and feeding to the model would cause a circular result |
| `next_30d_impressions_delta` | Excluded (leakage trap, Part 3) | calculated from the label window itself — demonstrated then removed |

## 3a. Verification query 1 — grain check
One row really is (report_date, client, content). Row count should equal the distinct-key count.

In [3]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_keys
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,9841378,9841378


**Read this cell's output before moving on.** If `total_rows` != `distinct_grain_keys`, the grain claim in Part 1 is wrong — fix the claim, don't ignore the mismatch.

## 3b. Verification query 2 — row count + date span

In [4]:
span_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
span_check

,row_count,min_date,max_date,distinct_clients,distinct_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


## 3c. Verification query 3 — availability (IS TRUE filter)
How many rows survive once you require actual GSC access on the client, joined from `dim_clients`.

In [7]:
for f in all_files:
    if "dim_clients" in f.lower():
        print(f)

dim_clients.parquet


In [8]:
dup_check = con.sql(f"""
    WITH daily AS (
        SELECT *
        FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
    ),
    one_key AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM daily
        GROUP BY 1,2,3
        HAVING COUNT(*) > 1
        LIMIT 1
    )
    SELECT d.*
    FROM daily d
    JOIN one_key k
      ON d.report_date = k.report_date
     AND d.client_hash_id = k.client_hash_id
     AND d.content_hash_id = k.content_hash_id
""").df()
dup_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month


In [11]:
availability_check = con.sql(f"""
    WITH daily AS (
        SELECT *
        FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
    ),
    clients AS (
        SELECT client_hash_id, has_gsc_access, has_ga4_access
        FROM read_parquet('{REL}/dim_clients.parquet')
    )
    SELECT
        (SELECT COUNT(*) FROM daily) AS rows_before_filter,
        COUNT(*) AS rows_after_gsc_filter
    FROM daily d
    JOIN clients c ON d.client_hash_id = c.client_hash_id
    WHERE c.has_gsc_access IS TRUE
""").df()
availability_check

,rows_before_filter,rows_after_gsc_filter
0,9841378,9829226


## 3d. Five features + "available when?" line

1. `impressions_90d` (log) — knowable at the decision moment because it's summed only over the prior 90-day window, not the label window.
2. `avg_position_90d` — knowable because it's averaged over the prior 90 days only.
3. `ctr_90d` — knowable because it's derived purely from prior-window impressions and clicks.
4. `days_since_last_update` — knowable because content-update metadata is static/backward-looking, not affected by future traffic.
5. `word_count` — knowable because it's a fixed content attribute at the decision point.

In [13]:
schema_fact = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
""").df()
print(schema_fact.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [14]:
for f in all_files:
    if "dim_content" in f.lower():
        print(f)

dim_content.parquet


In [15]:
schema_clients = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{REL}/dim_clients.parquet')
""").df()
print(schema_clients.to_string())

           column_name column_type null   key default extra
0       client_hash_id     VARCHAR  YES  None    None  None
1            is_active     BOOLEAN  YES  None    None  None
2       has_gsc_access     BOOLEAN  YES  None    None  None
3       has_ga4_access     BOOLEAN  YES  None    None  None
4       access_profile     VARCHAR  YES  None    None  None
5  client_created_date        DATE  YES  None    None  None
6  client_updated_date        DATE  YES  None    None  None
7       gsc_data_start        DATE  YES  None    None  None
8       ga4_data_start        DATE  YES  None    None  None


In [16]:
schema_content = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet')
""").df()
print(schema_content.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [17]:
print(dup_check.shape)
dup_check

(0, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month


In [18]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date::VARCHAR || '_' || client_hash_id || '_' || content_hash_id) AS distinct_grain_keys
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
""").df()
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,9841378,9841378


In [19]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows_before_filter,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_after_gsc_filter
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
""").df()
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_before_filter,rows_after_gsc_filter
0,9841378,3611061


In [20]:
feature_frame = con.sql(f"""
    WITH window_90d AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_90d,
            SUM(gsc_clicks) AS clicks_90d,
            AVG(gsc_avg_position) AS avg_position_90d
        FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        w.client_hash_id,
        w.content_hash_id,
        LOG(1 + w.impressions_90d) AS log_impressions_90d,
        w.avg_position_90d,
        CASE WHEN w.impressions_90d > 0 THEN w.clicks_90d::DOUBLE / w.impressions_90d ELSE NULL END AS ctr_90d,
        (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) AS days_since_last_update,
        d.word_count
    FROM window_90d w
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON w.content_hash_id = d.content_hash_id AND w.client_hash_id = d.client_hash_id
    WHERE w.impressions_90d > 0
""").df()
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,log_impressions_90d,avg_position_90d,ctr_90d,days_since_last_update,word_count
0,client_2094c6eb080311d5,content_14a3d47ccd0d15dc,0.477121,5.500000,0.000000,112,3864
1,client_2094c6eb080311d5,content_14a6f92117604fef,1.959041,7.936301,0.000000,-50,2867
2,client_2094c6eb080311d5,content_14a86c63a214f648,1.653213,30.479101,0.000000,112,2949
3,client_2094c6eb080311d5,content_14b1a02c1b8557fb,1.959041,28.426940,0.000000,110,4279
4,client_2094c6eb080311d5,content_14c17f59aa610ab3,2.089905,5.882498,0.016393,-83,3277


## 3e. The leakage trap

Deliberately add one label-derived column (built from the *next* 30 days — the outcome window), train a one-line quick classifier, watch the score jump toward perfect, then delete the column and keep the honest number.

In [22]:
label_window = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_next30
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE report_date <= DATE '2026-04-30'
    GROUP BY 1, 2
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [23]:
for f in all_files:
    if "fact_content_daily_performance" in f and "2026-04" in f:
        print(f)

fact_content_daily_performance/month=2026-04/data_0.parquet


In [26]:
# Build the label from the 30-day window AFTER the feature month, plus the leaky column
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

label_window = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_next30
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    WHERE report_date <= DATE '2026-04-30'
    GROUP BY 1, 2
""").df()

merged = feature_frame.merge(label_window, on=['client_hash_id', 'content_hash_id'], how='inner')
merged['future_decline'] = (merged['impressions_next30'] < 0.7 * (2**merged['log_impressions_90d'] - 1)).astype(int)

# THE TRAP: a column built directly from the label window
merged['next_30d_impressions_delta'] = merged['impressions_next30'] - (2**merged['log_impressions_90d'] - 1)

features_honest = ['log_impressions_90d', 'avg_position_90d', 'ctr_90d', 'word_count']
features_leaky = features_honest + ['next_30d_impressions_delta']

df = merged.dropna(subset=features_leaky + ['future_decline'])
X_train, X_test, y_train, y_test = train_test_split(df, df['future_decline'], test_size=0.3, random_state=42)

# Leaky version
clf_leaky = LogisticRegression(max_iter=1000).fit(X_train[features_leaky], y_train)
auc_leaky = roc_auc_score(y_test, clf_leaky.predict_proba(X_test[features_leaky])[:, 1])

# Honest version (leaky column removed)
clf_honest = LogisticRegression(max_iter=1000).fit(X_train[features_honest], y_train)
auc_honest = roc_auc_score(y_test, clf_honest.predict_proba(X_test[features_honest])[:, 1])

print(f"AUC WITH leaked column:    {auc_leaky:.3f}  <- inflated, worthless")
print(f"AUC WITHOUT leaked column: {auc_honest:.3f}  <- the honest number to keep")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

AUC WITH leaked column:    1.000  <- inflated, worthless
AUC WITHOUT leaked column: 0.921  <- the honest number to keep


**Note in your own words** (fill in after running): how big was the AUC gap, and why does `next_30d_impressions_delta` leak — it's built directly from the same window the label comes from, so the model isn't finding a real prior-window signal, it's just reading the answer off a near-copy of it.

## 4. Data limits

- **Unbalanced panel:** only 9 of 70 clients have 12+ months of daily history; most have far less, so seasonality comparisons aren't reliable for the majority of the panel.
- **GSC-only / GA4-only clients:** rows before a client's `ga4_data_start` carry search data only (`ga4_data_available = FALSE`) — a zero in a GA4 field there means "not tracked yet," not "zero traffic." Always check `dim_clients.gsc_data_start` / `ga4_data_start` before treating a gap as a real zero.
- **`source_only_missing_client_dimension` clients:** some client rows have no active/access flags at all — exclude or flag them explicitly rather than silently joining them in.
- **Window overlap risk:** the feature window (March) and label window (next 30 days) must never overlap — confirmed structurally above by using separate month partitions, but re-check this if you change the window definition later.

## 5. Self-check

- [ ] Every section filled — markdown thinking AND the code that backs it
- [ ] Notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to repo under `work/notebooks/w03_data_contract.ipynb` — then submit repo URL on the card